In [1]:
# @title 🚀 1. Cấu hình & Google Drive
GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/RAG-Document" # @param {type:"string"}
MODEL_REPO = "Qwen/Qwen2-7B-Instruct-GGUF" # @param {type:"string"}
MODEL_FILE = "qwen2-7b-instruct-q4_k_m.gguf" # @param {type:"string"}

import os
from google.colab import drive

try:
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive", force_remount=True)
    print("✅ Đã kết nối Google Drive.")
except Exception as e:
    print(f"❌ Lỗi kết nối Drive: {e}")

os.makedirs(GOOGLE_DRIVE_PATH, exist_ok=True)
os.chdir(GOOGLE_DRIVE_PATH)

❌ Lỗi kết nối Drive: mount failed


In [2]:
# @title 🛠️ Bước 1: Cài đặt hệ thống (Phiên bản tối ưu - Không bị treo)

# 1. Cài các thư viện tiện ích và server (Rất nhanh)
!pip install -q faiss-cpu pdfplumber fastapi uvicorn python-docx nest_asyncio opencv-python

# 2. Cài các thư viện AI có sẵn bản build (Torch, OCR)
!pip install -q easyocr sentence-transformers

# 3. Cài Llama-cpp-python bằng bản build sẵn cho CPU (Tiết kiệm 15-20 phút biên dịch)
!pip install -q llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/wheels/cpu

print("\n" + "="*30)
print("✅ Cài đặt hoàn tất! Hãy chuyển sang Bước 2.")
print("="*30)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 32.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 13.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Gett

In [3]:
# @title 🚀 3. Khởi tạo mô hình (Phiên bản đã sửa lỗi thiếu thư viện)
import os, subprocess
import json, shutil, threading, asyncio, time
import numpy as np
import faiss
import nest_asyncio
import torch, pdfplumber, easyocr, cv2
from PIL import Image
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama
from docx import Document

nest_asyncio.apply()
HAS_GPU = torch.cuda.is_available()

# --- CHOOSE MODEL BASED ON HARDWARE ---
if HAS_GPU:
    SELECTED_REPO = "Qwen/Qwen2-7B-Instruct-GGUF"
    SELECTED_FILE = "qwen2-7b-instruct-q4_k_m.gguf"
    print("🚀 Phát hiện GPU: Sử dụng mô hình lớn Qwen2-7B")
else:
    SELECTED_REPO = "Qwen/Qwen2-1.5B-Instruct-GGUF"
    SELECTED_FILE = "qwen2-1_5b-instruct-q4_k_m.gguf"
    print("🐢 Chạy CPU: Sử dụng mô hình nhẹ Qwen2-1.5B để tăng tốc")

model_path = os.path.join(GOOGLE_DRIVE_PATH, SELECTED_FILE)

if not os.path.exists(model_path):
    print(f"📥 Đang tải mô hình {SELECTED_FILE}...")
    !wget -q --show-progress -c https://huggingface.co/{SELECTED_REPO}/resolve/main/{SELECTED_FILE} -O {model_path}

print("📦 Loading Sentence-Transformer (MiniLM-L12-v2 - Siêu nhanh cho CPU)...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print(f"📷 Initializing OCR ({'GPU' if HAS_GPU else 'CPU'})...")
ocr_reader = easyocr.Reader(["vi", "en"], gpu=HAS_GPU)

print(f"🧠 Loading LLM ({'GPU' if HAS_GPU else 'CPU'})...")
if HAS_GPU:
    llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=4096, verbose=True) # Increased n_ctx and added verbose=True
else:
    llm = Llama(model_path=model_path, n_gpu_layers=0, n_ctx=4096, n_threads=4, verbose=True) # Increased n_ctx and added verbose=True

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

vector_db = None
chunks = []

# Cấu hình tối ưu theo môi trường
CHUNK_SIZE = 400 if HAS_GPU else 200
CHUNK_OVERLAP = 50 if HAS_GPU else 30
TOP_K = 2 if HAS_GPU else 1
MAX_TOKENS = 1024 if HAS_GPU else 256

print(f"\n{'='*50}")
print(f"✅ Sẵn sàng! Chế độ: {'🚀 GPU' if HAS_GPU else '🐢 CPU'}")
print(f"   Model: {SELECTED_FILE} | Chunk: {CHUNK_SIZE} | Top-K: {TOP_K}")
print(f"{'='*50}")

🚀 Phát hiện GPU: Sử dụng mô hình lớn Qwen2-7B
📥 Đang tải mô hình qwen2-7b-instruct-q4_k_m.gguf...
/content/drive/MyDr 100%[===================>]   4.36G   152MB/s    in 29s     
📦 Loading Sentence-Transformer (MiniLM-L12-v2 - Siêu nhanh cho CPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Progress: |██████████████████████████████████████████████████| 100.0% Complete

llama_model_loader: loaded meta data with 26 key-value pairs and 339 tensors from /content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.name str              = qwen2-7b-instruct
llama_model_loader: - kv   2:                          qwen2.block_count u32              = 28
llama_model_loader: - kv   3:                       qwen2.context_length u32              = 32768
llama_model_loader: - kv   4:                     qwen2.embedding_length u32              = 3584
llama_model_loader: - kv   5:                  qwen2.feed_forward_length u32              = 18944
llama_model_loader: - kv   6:                 qwen2.attention.head_count u32              = 28
llama_model_loader: - kv   7:       

🧠 Loading LLM (GPU)...


llama_model_loader: - kv  13:                      tokenizer.ggml.tokens arr[str,152064]  = ["!", "\"", "#", "$", "%", "&", "'", ...
llama_model_loader: - kv  14:                  tokenizer.ggml.token_type arr[i32,152064]  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
llama_model_loader: - kv  15:                      tokenizer.ggml.merges arr[str,151387]  = ["Ġ Ġ", "ĠĠ ĠĠ", "i n", "Ġ t",...
llama_model_loader: - kv  16:                tokenizer.ggml.eos_token_id u32              = 151645
llama_model_loader: - kv  17:            tokenizer.ggml.padding_token_id u32              = 151643
llama_model_loader: - kv  18:                tokenizer.ggml.bos_token_id u32              = 151643
llama_model_loader: - kv  19:                    tokenizer.chat_template str              = {% for message in messages %}{% if lo...
llama_model_loader: - kv  20:               tokenizer.ggml.add_bos_token bool             = false
llama_model_loader: - kv  21:               general.quantization_version u32    


✅ Sẵn sàng! Chế độ: 🚀 GPU
   Model: qwen2-7b-instruct-q4_k_m.gguf | Chunk: 400 | Top-K: 2


In [15]:
# @title 🚀 3. Khởi tạo mô hình (Tự nhận diện GPU/CPU & Model)
import json, shutil, threading, asyncio, os, subprocess, time
import numpy as np
import faiss
import nest_asyncio
import torch, pdfplumber, easyocr, cv2
from PIL import Image
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama
from docx import Document

nest_asyncio.apply()
HAS_GPU = torch.cuda.is_available()
SELECTED_REPO = "Qwen/Qwen2-7B-Instruct-GGUF"
SELECTED_FILE = "qwen2-7b-instruct-q4_k_m.gguf"
print("🐢 Chạy CPU: Sử dụng mô hình nhẹ Qwen2-1.5B để tăng tốc")
# # --- CHOOSE MODEL BASED ON HARDWARE ---
# if HAS_GPU:
#     SELECTED_REPO = "Qwen/Qwen2-7B-Instruct-GGUF"
#     SELECTED_FILE = "qwen2-7b-instruct-q4_k_m.gguf"
#     print("🚀 Phát hiện GPU: Sử dụng mô hình lớn Qwen2-7B")
# else:
#     SELECTED_REPO = "Qwen/Qwen2-1.5B-Instruct-GGUF"
#     SELECTED_FILE = "qwen2-1_5b-instruct-q4_k_m.gguf"
#     print("🐢 Chạy CPU: Sử dụng mô hình nhẹ Qwen2-1.5B để tăng tốc")

model_path = os.path.join(GOOGLE_DRIVE_PATH, SELECTED_FILE)

if not os.path.exists(model_path):
    print(f"📥 Đang tải mô hình {SELECTED_FILE}...")
    !wget -q --show-progress -c https://huggingface.co/{SELECTED_REPO}/resolve/main/{SELECTED_FILE} -O {model_path}

print("📦 Loading Sentence-Transformer (MiniLM-L12-v2 - Siêu nhanh cho CPU)...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print(f"📷 Initializing OCR ({'GPU' if HAS_GPU else 'CPU'})...")
ocr_reader = easyocr.Reader(["vi", "en"], gpu=HAS_GPU)

print(f"🧠 Loading LLM ({'GPU' if HAS_GPU else 'CPU'})...")
if HAS_GPU:
    llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=4096) # Increased n_ctx
else:
    llm = Llama(model_path=model_path, n_gpu_layers=0, n_ctx=4096, n_threads=4) # Increased n_ctx

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

vector_db = None
chunks = []

# Cấu hình tối ưu theo môi trường
CHUNK_SIZE = 400 if HAS_GPU else 200
CHUNK_OVERLAP = 50 if HAS_GPU else 30
TOP_K = 2 if HAS_GPU else 1
MAX_TOKENS = 1024 if HAS_GPU else 256

print(f"\n{'='*50}")
print(f"✅ Sẵn sàng! Chế độ: {'🚀 GPU' if HAS_GPU else '🐢 CPU'}")
print(f"   Model: {SELECTED_FILE} | Chunk: {CHUNK_SIZE} | Top-K: {TOP_K}")
print(f"{'='*50}")


🐢 Chạy CPU: Sử dụng mô hình nhẹ Qwen2-1.5B để tăng tốc
📦 Loading Sentence-Transformer (MiniLM-L12-v2 - Siêu nhanh cho CPU)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📷 Initializing OCR (GPU)...


llama_model_loader: loaded meta data with 26 key-value pairs and 339 tensors from /content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.name str              = qwen2-7b-instruct
llama_model_loader: - kv   2:                          qwen2.block_count u32              = 28
llama_model_loader: - kv   3:                       qwen2.context_length u32              = 32768
llama_model_loader: - kv   4:                     qwen2.embedding_length u32              = 3584
llama_model_loader: - kv   5:                  qwen2.feed_forward_length u32              = 18944
llama_model_loader: - kv   6:                 qwen2.attention.head_count u32              = 28
llama_model_loader: - kv   7:       

🧠 Loading LLM (GPU)...


init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
load: control token: 151644 '<|im_start|>' is not marked as EOG
load: printing all EOG tokens:
load:   - 128247 ('</s>')
load:   - 151643 ('<|endoftext|>')
load:   - 151645 ('<|im_end|>')
load: special tokens cache size = 422
load: token to piece cache size = 0.9352 MB
print_info: arch                  = qwen2
print_info: vocab_only            = 0
print_info: no_alloc              = 0
print_info: n_ctx_train           = 32768
print_info: n_embd                = 3584
print_info: n_embd_inp            = 3584
print_info: n_layer               = 28
print_info: n_head                = 28
print_info: n_head_kv             = 4
print_info: n_rot                 = 128
print_info: n_swa                 = 0
print_info: is_swa_any            = 0
print_info: n_embd_head_k         = 128
print_info: n_


✅ Sẵn sàng! Chế độ: 🚀 GPU
   Model: qwen2-7b-instruct-q4_k_m.gguf | Chunk: 400 | Top-K: 2


In [16]:
# @title 🚀 4. Logic xử lý RAG
def save_state():
    with open("chunks_metadata.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)
    if vector_db is not None:
        faiss.write_index(vector_db, "faiss_index.bin")

def rebuild_index_from_scratch():
    global vector_db
    if not chunks:
        vector_db = None
        if os.path.exists("faiss_index.bin"): os.remove("faiss_index.bin")
        return
    print(f"🔄 Đang xây dựng lại mục lục cho {len(chunks)} đoạn văn...")
    texts = [c["text"] for c in chunks]
    embeddings = np.array(embed_model.encode(texts)).astype("float32")
    vector_db = faiss.IndexFlatL2(embeddings.shape[1])
    vector_db.add(embeddings)
    save_state()

def add_to_index(new_meta_chunks):
    global vector_db, chunks
    if not new_meta_chunks: return
    try:
        print(f"⚡ Đang Embedding {len(new_meta_chunks)} đoạn văn mới (Siêu tốc)...")
        new_embeddings = np.array(embed_model.encode([c["text"] for c in new_meta_chunks])).astype("float32")

        if vector_db is None:
            dim = new_embeddings.shape[1]
            vector_db = faiss.IndexFlatL2(dim)

        vector_db.add(new_embeddings)
        chunks.extend(new_meta_chunks)
        save_state()
        print(f"✅ Đã thêm {len(new_meta_chunks)} đoạn mới. Tổng cộng: {len(chunks)} đoạn.")
        print("✅ Đã cập nhật bộ nhớ RAG thành công.")
    except Exception as e:
        print(f"❌ Lỗi Embedding: {e}")

def table_to_markdown(table):
    if not table: return ""
    markdown = "\n[Dữ liệu Bảng]\n"
    for row in table:
        markdown += "| " + " | ".join([str(cell).strip().replace("\n", " ") if cell else "" for cell in row]) + " |\n"
    return markdown

def extract_text(file_path):
    text = ""
    try:
        ext = os.path.splitext(file_path)[1].lower()
        if ext == ".pdf":
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    p_text = page.extract_text() or ""
                    tables = page.extract_tables()
                    for table in tables:
                        p_text += "\n" + table_to_markdown(table)
                    if len(p_text.strip()) < 50:
                        img = page.to_image().original.convert("RGB")
                        ocr_res = ocr_reader.readtext(np.array(img), detail=0)
                        p_text += " " + " ".join(ocr_res)
                    text += p_text + "\n"
        elif ext == ".docx":
            text = " ".join([p.text for p in Document(file_path).paragraphs])
        elif ext in [".png", ".jpg", ".jpeg"]:
            ocr_res = ocr_reader.readtext(file_path, detail=0)
            text = " ".join(ocr_res)
    except Exception as e:
        print(f"Lỗi khi đọc file {file_path}: {e}")
    return text

def split_text(text, size=None, overlap=None):
    size = size or CHUNK_SIZE
    overlap = overlap or CHUNK_OVERLAP
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size-overlap)]


In [17]:
# @title 🚀 5. API Endpoints
class ChatRequest(BaseModel): query: str
class DeleteRequest(BaseModel): filename: str

@app.get("/files")
async def get_files():
    return {"files": [f for f in os.listdir(GOOGLE_DRIVE_PATH) if f.lower().endswith((".pdf", ".docx", ".png", ".jpg", ".jpeg"))]}

@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    global vector_db, chunks
    target_path = os.path.join(GOOGLE_DRIVE_PATH, file.filename)
    with open(target_path, "wb") as b: shutil.copyfileobj(file.file, b)

    text = extract_text(target_path)
    if not text: raise HTTPException(status_code=400, detail="Không thể trích xuất văn bản")

    new_text_chunks = split_text(text)
    new_meta_chunks = [{"text": t, "filename": file.filename} for t in new_text_chunks]

    # Dùng add_to_index cho file mới tải lên
    add_to_index(new_meta_chunks)
    return {"status": "success", "total_chunks": len(chunks)}

@app.post("/delete_file")
async def delete_file(req: DeleteRequest):
    global chunks
    file_path = os.path.join(GOOGLE_DRIVE_PATH, req.filename)
    if os.path.exists(file_path): os.remove(file_path)

    initial_count = len(chunks)
    chunks = [c for c in chunks if c["filename"] != req.filename]
    if len(chunks) != initial_count:
        # Sửa ở đây: Khi xóa file phải xây dựng lại toàn bộ Index
        rebuild_index_from_scratch()
    return {"status": "success", "deleted": req.filename, "remaining_chunks": len(chunks)}

@app.post("/clear")
async def clear_data():
    global vector_db, chunks
    vector_db, chunks = None, []
    if os.path.exists("faiss_index.bin"): os.remove("faiss_index.bin")
    if os.path.exists("chunks_metadata.json"): os.remove("chunks_metadata.json")
    for f in os.listdir(GOOGLE_DRIVE_PATH):
        if f.lower().endswith((".pdf", ".docx", ".png", ".jpg", ".jpeg")):
            os.remove(os.path.join(GOOGLE_DRIVE_PATH, f))
    return {"message": "Đã xóa sạch bộ nhớ RAG."}

@app.post("/chat")
async def chat(req: ChatRequest):
    print(f"--- NHẬN CÂU HỎI: {req.query} ---")
    if vector_db is None or not chunks:
        return StreamingResponse(iter(["Bạn chưa tải tài liệu nào lên bộ nhớ trợ lý!"]), media_type="text/plain")

    try:
        query_emb = embed_model.encode([req.query])
        D, I = vector_db.search(np.array(query_emb).astype("float32"), k=TOP_K)

        retrieved_meta = [chunks[idx] for idx in I[0] if idx < len(chunks)]
        context = "\n---\n".join([c["text"] for c in retrieved_meta])

        prompt = f"<|im_start|>system\nTrả lời bằng tiếng Việt ngắn gọn, súc tích.\nNgữ cảnh:\n{context}<|im_end|>\n<|im_start|>user\n{req.query}<|im_end|>\n<|im_start|>assistant\n"

        async def stream_generator():
            print("[DEBUG STREAM] stream_generator started.")
            # Ensure context and prompt are not empty
            if not context: print("[DEBUG STREAM] Context is EMPTY!")
            if not req.query: print("[DEBUG STREAM] Query is EMPTY!")
            print(f"[DEBUG STREAM] Prompt length: {len(prompt)} characters.")
            # Uncomment the line below to print the full prompt for debugging
            print(f"[DEBUG STREAM] Full Prompt:\n{prompt}")

            output = llm(prompt, max_tokens=MAX_TOKENS, stop=["<|im_end|>"] , stream=True)
            print(f"[DEBUG STREAM] Type of LLM output: {type(output)}")
            if not output: # This checks if the object itself is falsy, not if the generator is empty.
                print("[DEBUG STREAM] LLM returned a falsy output object.")
                yield "Lỗi: Mô hình không tạo ra phản hồi nào (đối tượng đầu ra rỗng)."
                return

            print("[DEBUG STREAM] LLM output iterator obtained. Attempting to yield tokens.")
            generated_any_token = False
            try:
                for i, chunk in enumerate(output):
                    print(f"[DEBUG STREAM] Processing chunk {i}: {chunk}")
                    token = chunk["choices"][0]["text"]
                    if not token: # If token is empty, continue to next chunk, or break if it's a known end
                        print(f"[DEBUG STREAM] Received empty token from LLM for chunk: {chunk}")
                        continue # Try to get next chunk
                    print(f"[DEBUG STREAM] Token: '{token}'")
                    generated_any_token = True
                    if "<|" in token or "im_end" in token:
                        print(f"[DEBUG STREAM] Detected stop sequence in token: '{token}'. Breaking loop.")
                        break
                    yield token
            except Exception as iter_e:
                print(f"[DEBUG STREAM] Exception during iteration over LLM output: {iter_e}")
                yield f"Lỗi khi xử lý phản hồi từ mô hình: {iter_e}"
                return

            if not generated_any_token:
                print("[DEBUG STREAM] LLM did not generate any visible tokens before finishing or encountering stop condition.")
                yield "Lỗi: Mô hình không tạo ra phản hồi có nghĩa nào."

            print("[DEBUG STREAM] stream_generator finished processing LLM output.")

            sources = list(set([c["filename"] for c in retrieved_meta]))
            if sources:
                source_text = f"\n\n---\n📌 Nguồn: {', '.join(sources)}"
                print(f"[DEBUG STREAM] Yielding sources: {source_text}")
                yield source_text
            else:
                print("[DEBUG STREAM] No sources to yield.")

        return StreamingResponse(stream_generator(), media_type="text/plain")
    except Exception as e:
        print(f"❌ Lỗi Chat: {e}")
        return StreamingResponse(iter([f"Lỗi hệ thống: {e}!"]), media_type="text/plain")


In [18]:
# @title 🛠️ Cài đặt Cloudflared (nếu chưa có)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version
print("✅ Cloudflared đã được cài đặt!")

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)
✅ Cloudflared đã được cài đặt!


In [ ]:
# @title 🚀 6. Chạy Server
import subprocess, threading, uvicorn, time

def run_tunnel():
    import subprocess
    subprocess.run(["pkill", "cloudflared"], capture_output=True)
    time.sleep(2)
    # Changed 'cloudflared' to './cloudflared' to specify the path
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if ".trycloudflare.com" in line:
            url = [p for p in line.split() if 'https://' in p][0]
            print(f"\n{'='*50}")
            print(f"✅ ĐÃ SẴN SÀNG! Chế độ: {'🚀 GPU' if HAS_GPU else '🐢 CPU'}")
            print(f"🔗 URL: {url}")
            print(f"👉 Copy link trên dán vào giao diện Chat")
            print(f"{'='*50}\n")
            break

# Giải phóng port 8000 triệt để
!fuser -k 8000/tcp 2>/dev/null
!lsof -ti:8000 | xargs kill -9 2>/dev/null
import time; time.sleep(3)  # Đợi 3 giây cho port được giải phóng hoàn toàn

threading.Thread(target=run_tunnel, daemon=True).start()

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
await uvicorn.Server(config).serve()

INFO:     Started server process [2922]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



✅ ĐÃ SẴN SÀNG! Chế độ: 🚀 GPU
🔗 URL: https://whole-minimize-custom-lotus.trycloudflare.com
👉 Copy link trên dán vào giao diện Chat

INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "GET /files HTTP/1.1" 200 OK
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "GET /files HTTP/1.1" 200 OK
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "OPTIONS /upload HTTP/1.1" 200 OK
⚡ Đang Embedding 1 đoạn văn mới (Siêu tốc)...
✅ Đã thêm 1 đoạn mới. Tổng cộng: 1 đoạn.
✅ Đã cập nhật bộ nhớ RAG thành công.
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "GET /files HTTP/1.1" 200 OK
⚡ Đang Embedding 1 đoạn văn mới (Siêu tốc)...
✅ Đã thêm 1 đoạn mới. Tổng cộng: 2 đoạn.
✅ Đã cập nhật bộ nhớ RAG thành công.
INFO:     2405:48

llama_perf_context_print:        load time =  105449.31 ms
llama_perf_context_print: prompt eval time =  105440.17 ms /   538 tokens (  195.99 ms per token,     5.10 tokens per second)
llama_perf_context_print:        eval time =  114294.93 ms /   168 runs   (  680.33 ms per token,     1.47 tokens per second)
llama_perf_context_print:       total time =  219988.36 ms /   706 tokens
llama_perf_context_print:    graphs reused =        167


[DEBUG STREAM] Processing chunk 168: {'id': 'cmpl-be2c43e2-97f2-4bb7-a8eb-51191913f594', 'object': 'text_completion', 'created': 1775371943, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}]}
[DEBUG STREAM] Received empty token from LLM for chunk: {'id': 'cmpl-be2c43e2-97f2-4bb7-a8eb-51191913f594', 'object': 'text_completion', 'created': 1775371943, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}]}
[DEBUG STREAM] stream_generator finished processing LLM output.
[DEBUG STREAM] Yielding sources: 

---
📌 Nguồn: technical_specifications.pdf, BieuDoSanLuongLuongThucNga.png
--- NHẬN CÂU HỎI: Sản lượng lương thực năm 2005 tăng hay giảm bao nhiêu triệu tấn so với năm 1995? ---
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "POST /chat HTTP/1.1" 200 OK
[DEBUG STREAM] strea

Llama.generate: 495 prefix-match hit, remaining 33 prompt tokens to eval


...

[DEBUG STREAM] Processing chunk 34: {'id': 'cmpl-e1921a42-e598-4990-a451-c3d91551587f', 'object': 'text_completion', 'created': 1775372556, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}]}
[DEBUG STREAM] Received empty token from LLM for chunk: {'id': 'cmpl-e1921a42-e598-4990-a451-c3d91551587f', 'object': 'text_completion', 'created': 1775372556, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}]}
[DEBUG STREAM] stream_generator finished processing LLM output.
[DEBUG STREAM] Yielding sources: 

---
📌 Nguồn: technical_specifications.pdf, BieuDoSanLuongLuongThucNga.png
--- NHẬN CÂU HỎI: Công suất định mức và trọng lượng của Model Advanced (A2) là bao nhiêu? ---
INFO:     2405:4803:525c:baa0:24c1:6918:3aab:a56e:0 - "POST /chat HTTP/1.1" 200 OK
[DEBUG STREAM] stream_generato

Llama.generate: 20 prefix-match hit, remaining 500 prompt tokens to eval


[DEBUG STREAM] Processing chunk 0: {'id': 'cmpl-7f806683-439e-4c7d-83cf-844a1a609543', 'object': 'text_completion', 'created': 1775372681, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': 'C', 'index': 0, 'logprobs': None, 'finish_reason': None}]}
[DEBUG STREAM] Token: 'C'
[DEBUG STREAM] Processing chunk 1: {'id': 'cmpl-7f806683-439e-4c7d-83cf-844a1a609543', 'object': 'text_completion', 'created': 1775372681, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': 'ông', 'index': 0, 'logprobs': None, 'finish_reason': None}]}
[DEBUG STREAM] Token: 'ông'
[DEBUG STREAM] Processing chunk 2: {'id': 'cmpl-7f806683-439e-4c7d-83cf-844a1a609543', 'object': 'text_completion', 'created': 1775372681, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': ' suất', 'index': 0, 'logprobs': None, 'finish_reason': None}]}
[DEBUG STREAM] Token: ' suất'
[DEBUG STREAM] P

llama_perf_context_print:        load time =  105449.31 ms
llama_perf_context_print: prompt eval time =   96951.73 ms /   500 tokens (  193.90 ms per token,     5.16 tokens per second)
llama_perf_context_print:        eval time =   19555.65 ms /    28 runs   (  698.42 ms per token,     1.43 tokens per second)
llama_perf_context_print:       total time =  116546.81 ms /   528 tokens
llama_perf_context_print:    graphs reused =         27


[DEBUG STREAM] Processing chunk 28: {'id': 'cmpl-7f806683-439e-4c7d-83cf-844a1a609543', 'object': 'text_completion', 'created': 1775372681, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}]}
[DEBUG STREAM] Received empty token from LLM for chunk: {'id': 'cmpl-7f806683-439e-4c7d-83cf-844a1a609543', 'object': 'text_completion', 'created': 1775372681, 'model': '/content/drive/MyDrive/RAG-Document/qwen2-7b-instruct-q4_k_m.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}]}
[DEBUG STREAM] stream_generator finished processing LLM output.
[DEBUG STREAM] Yielding sources: 

---
📌 Nguồn: technical_specifications.pdf, BieuDoSanLuongLuongThucNga.png
